## **TRAINING NB**

1. **Annotations** from VisDrone-MOT dataset are read and converted to YOLO format (`class_id x_center y_center width height`).
2. **Images and labels** are saved into training (`train/`) and validation (`val/`) sets based on the `VAL_SPLIT` ratio.
3. Small objects (below `MIN_BOX` size) and unwanted classes (defined by `IGNORE_CLASS`) are filtered out.

### Output:
- Images and labels in YOLO format, ready for training.

### Parameters:
- `FRAME_SKIP`: Process every Nth frame.
- `VAL_SPLIT`: Split ratio for training/validation.
- `IGNORE_CLASS`: Class to ignore during conversion.
- `MIN_BOX`: Minimum bounding box size to keep.


In [ ]:
import os
import cv2
import random

# =========================
# VisDrone Dataset Conversion to YOLO format
# =========================

# Ruta de las carpetas del dataset original VisDrone y la carpeta de salida para YOLO
VISDRONE_ROOT = "datasets/VisDrone-MOT"
OUTPUT_ROOT = "datasets/visdrone_yolo"

# Configuración de parámetros
FRAME_SKIP = 10  # Procesar solo cada N-ésimo frame (para reducir la cantidad de datos)
VAL_SPLIT = 0.15  # Proporción de datos para el conjunto de validación

IGNORE_CLASS = 0  # Clase a ignorar (por ejemplo, fondo)

# Tamaño mínimo del objeto (muy importante para no perder objetos pequeños)
MIN_BOX = 8  # Los objetos más pequeños que este valor serán ignorados

# =========================

# Crear las carpetas para guardar las imágenes y las etiquetas en formato YOLO
img_train = os.path.join(OUTPUT_ROOT, "images/train")
img_val = os.path.join(OUTPUT_ROOT, "images/val")
lab_train = os.path.join(OUTPUT_ROOT, "labels/train")
lab_val = os.path.join(OUTPUT_ROOT, "labels/val")

# Crear las carpetas si no existen
for p in [img_train, img_val, lab_train, lab_val]:
    os.makedirs(p, exist_ok=True)

# Directorios donde se encuentran las secuencias de imágenes y las anotaciones
sequences_path = os.path.join(VISDRONE_ROOT, "sequences")
annotations_path = os.path.join(VISDRONE_ROOT, "annotations")

# Contador de imágenes procesadas
image_counter = 0

# Iterar a través de las secuencias de imágenes
for seq in os.listdir(sequences_path):

    # Definir las rutas de cada secuencia y el archivo de anotaciones correspondiente
    seq_dir = os.path.join(sequences_path, seq)
    ann_file = os.path.join(annotations_path, seq + ".txt")

    # Si el archivo de anotaciones no existe, continuar con la siguiente secuencia
    if not os.path.exists(ann_file):
        continue

    print("Processing:", seq)

    # Diccionario para almacenar las anotaciones por frame
    annotations = {}

    # Leer el archivo de anotaciones y procesar línea por línea
    with open(ann_file) as f:
        for line in f:

            # Dividir la línea en partes (basado en coma)
            parts = line.strip().split(",")

            # Extraer los valores de las anotaciones
            frame_id = int(parts[0])
            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])
            cls = int(parts[7])

            # Ignorar la clase de objeto especificada para ser ignorada
            if cls == IGNORE_CLASS:
                continue

            # Convertir la clase VisDrone a la clase YOLO (VisDrone comienza desde 1, YOLO desde 0)
            cls = cls - 1

            # Asegurar que la clase esté en el rango válido
            if cls < 0 or cls > 9:
                continue

            # Filtrar objetos demasiado pequeños (en términos de área de la caja delimitadora)
            if w < MIN_BOX or h < MIN_BOX:
                continue

            # Almacenar las anotaciones para el frame actual
            annotations.setdefault(frame_id, []).append((x, y, w, h, cls))

    # Obtener las imágenes de la secuencia y ordenarlas
    frames = sorted(os.listdir(seq_dir))

    # Procesar cada imagen en la secuencia
    for i, frame_name in enumerate(frames):

        # Saltar frames según el valor de FRAME_SKIP
        if i % FRAME_SKIP != 0:
            continue

        # Obtener el ID del frame a partir del nombre del archivo
        frame_id = int(frame_name.split(".")[0])

        # Definir la ruta de la imagen
        img_path = os.path.join(seq_dir, frame_name)
        img = cv2.imread(img_path)

        # Si no se pudo leer la imagen, continuar con la siguiente
        if img is None:
            continue

        # Obtener las dimensiones de la imagen
        H, W = img.shape[:2]

        # Lista para almacenar las líneas de etiquetas en formato YOLO
        label_lines = []

        # Iterar sobre las anotaciones del frame actual y convertirlas al formato YOLO
        for (x, y, w, h, cls) in annotations.get(frame_id, []):

            # Convertir las coordenadas de la caja delimitadora al formato YOLO (normalizado)
            xc = (x + w / 2) / W
            yc = (y + h / 2) / H
            w_norm = w / W
            h_norm = h / H

            # Crear la línea de etiqueta para este objeto
            label_lines.append(f"{cls} {xc} {yc} {w_norm} {h_norm}")

        # Si no hay etiquetas para este frame, continuar con el siguiente
        if len(label_lines) == 0:
            continue

        # Decidir si esta imagen va al conjunto de entrenamiento o validación (aleatoriamente)
        if random.random() < VAL_SPLIT:
            img_out = os.path.join(img_val, f"{seq}_{frame_name}")
            lab_out = os.path.join(lab_val, f"{seq}_{frame_name.replace('.jpg','.txt')}")
        else:
            img_out = os.path.join(img_train, f"{seq}_{frame_name}")
            lab_out = os.path.join(lab_train, f"{seq}_{frame_name.replace('.jpg','.txt')}")

        # Guardar la imagen procesada
        cv2.imwrite(img_out, img)

        # Guardar las etiquetas en el archivo correspondiente
        with open(lab_out, "w") as f:
            f.write("\n".join(label_lines))

        # Incrementar el contador de imágenes procesadas
        image_counter += 1

# Imprimir el número total de imágenes procesadas
print("Dataset generated")
print("Total images:", image_counter)

Procesando: uav0000013_00000_v
Procesando: uav0000013_01073_v
Procesando: uav0000013_01392_v
Procesando: uav0000020_00406_v
Procesando: uav0000071_03240_v
Procesando: uav0000072_04488_v
Procesando: uav0000072_05448_v
Procesando: uav0000072_06432_v
Procesando: uav0000076_00720_v
Procesando: uav0000079_00480_v
Procesando: uav0000084_00000_v
Procesando: uav0000099_02109_v
Procesando: uav0000124_00944_v
Procesando: uav0000126_00001_v
Procesando: uav0000138_00000_v
Procesando: uav0000140_01590_v
Procesando: uav0000143_02250_v
Procesando: uav0000145_00000_v
Procesando: uav0000150_02310_v
Procesando: uav0000218_00001_v
Procesando: uav0000222_03150_v
Procesando: uav0000239_03720_v
Procesando: uav0000239_12336_v
Procesando: uav0000243_00001_v
Procesando: uav0000244_01440_v
Procesando: uav0000248_00001_v
Procesando: uav0000263_03289_v
Procesando: uav0000264_02760_v
Procesando: uav0000266_03598_v
Procesando: uav0000266_04830_v
Procesando: uav0000270_00001_v
Procesando: uav0000273_00001_v
Procesan

In [1]:
# 1. Instalación de dependencias (Ejecutar solo si es necesario)
# !pip install ultralytics torch torchvision

import torch
from datetime import datetime
from ultralytics import YOLO
import os

# ==========================================
# 2. DETECCIÓN Y CONFIGURACIÓN DE GPU
# ==========================================
def check_device():
    if torch.cuda.is_available():
        n_gpus = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        print(f"✅ GPU DETECTADA: {gpu_name}")
        print(f"Total de GPUs disponibles: {n_gpus}")
        print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
        device = 0  # Usar la primera GPU (ID 0)
    else:
        print("No se detectó GPU. Se usará la CPU (Esto será muy lento).")
        device = 'cpu'
    return device

selected_device = check_device()

✅ GPU DETECTADA: NVIDIA RTX 2000 Ada Generation
Total de GPUs disponibles: 1
VRAM Total: 17.18 GB


In [ ]:
dataset_path = "./datasets/visdrone_yolo"

print("Carpetas en dataset:")
print(os.listdir(dataset_path))

train_images = os.listdir(".datasets/visdrone_yolo/images/train")
print("Número de imágenes train:", len(train_images))

Carpetas en dataset:
['data.yaml', 'images', 'labels']
Número de imágenes train: 2068


### **MODEL CONFIGURATION AND HYPERPARAMETERS**

This block of code configures the main parameters for training a **YOLO** model on the **VisDrone-MOT** dataset.

### 1. **Model Weights**:
This specifies which YOLO model will be used. You can choose between different versions depending on the desired model size and speed:
- `yolo11m.pt`: Medium model.
- `yolo11s.pt`: Small model (faster, less accurate).
- `yolo11l.pt`: Large model (more accurate, slower).

```python
MODEL_WEIGHTS = "yolo11m.pt"  # Change to yolo11s.pt / yolo11l.pt as needed

In [ ]:
# --- CONFIGURACIÓN DE MODELO E HIPERPARAMETROS ---
MODEL_WEIGHTS = "yolo11m.pt"       # Cambia a yolo11s.pt / yolo11l.pt según necesidad
PROJECT_DIR   = "runs/train_visdrone"
RUN_NAME      = f"drone_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

train_params = {
    'data': './datasets/visdrone_yolo/data.yaml',
    'epochs': 150,            # Más épocas porque el dataset es pequeño, pero con paciencia.
    'patience': 25,           # Si no mejora en 50 épocas, se detiene solo (evita overfitting).
    'imgsz': 960,             # SUBIR A 800: En drones, los objetos son pequeños. 800 ayuda mucho más que 640.
    'batch': 24,              # Ajustar según la memoria de tu RTX 2000 (puedes probar 16 o 32).
    'device': 0, 
    'optimizer': 'AdamW',     # AdamW suele converger mejor en datasets pequeños.
    'lr0': 0.0008,             # Tasa de aprendizaje inicial.
    'lrf': 0.01,              # Tasa de aprendizaje final (coseno).
    'weight_decay': 0.0005,   # Regularización para que el modelo no se "aprenda de memoria" las fotos.

    # --- ESTRATEGIA DE AUMENTACIÓN AGRESIVA (Clave para 250 fotos) ---
    'mosaic': 1.0,            # Mezcla 4 imágenes en una. Fundamental para detectar objetos pequeños.
    'mixup': 0.1,             # Mezcla dos imágenes diferentes. Crea "nuevos" ejemplos sintéticos.
    'copy_paste': 0.1,        # "Pega" objetos de una imagen en otra. Excelente para aumentar ejemplos de personas.
    'degrees': 90.0,          # Rotación total. En drones, un vehículo puede verse en cualquier ángulo (0-360°).
    'scale': 0.5,             # Simula cambios de altura del drone (zoom in/out).
    'shear': 2.0,             # Distorsión de perspectiva.
    'perspective': 0.0001,    # Simula cambios en el ángulo de la cámara.
    'flipud': 0.5,            # Volteo vertical (necesario en cámaras cenitales).
    'fliplr': 0.5,            # Volteo horizontal.
    'hsv_h': 0.015,           # Variación de tono (para cambios de color de vehículos).
    'hsv_s': 0.7,             # Variación de saturación.
    'hsv_v': 0.4,             # Variación de valor (BRILLO - Crucial para nubosidad/sombras).
    
    'rect': True,
    'close_mosaic': 10,
    
    # Salida
    "plots": True,
    "project": PROJECT_DIR,
    "name": RUN_NAME,
    "exist_ok": False,
    "verbose": True,
    "save": True,
    "save_period": 25,   # Checkpoint cada 50 épocas
}

In [ ]:
# --- CALLBACK ETA ---
class ETACallback:
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs
        self.start_time = None

    def on_train_start(self, trainer):
        self.start_time = time.time()

    def on_fit_epoch_end(self, trainer):
        epoch = trainer.epoch + 1
        elapsed = time.time() - self.start_time

        time_per_epoch = elapsed / epoch
        remaining_epochs = self.total_epochs - epoch
        eta = time_per_epoch * remaining_epochs

        eta_min = int(eta // 60)
        eta_sec = int(eta % 60)

        print(f"⏳ ETA: {eta_min}m {eta_sec}s")


# --- ENTRENAMIENTO ---
def main():
    print(f"Iniciando entrenamiento: {RUN_NAME}")
    print(f"Modelo base : {MODEL_WEIGHTS}")
    print(f"Resultados  : {PROJECT_DIR}/{RUN_NAME}\n")

    model = YOLO(MODEL_WEIGHTS)

    # 👉 añadir ETA
    eta_cb = ETACallback(train_params["epochs"])
    model.add_callback("on_train_start", eta_cb.on_train_start)
    model.add_callback("on_fit_epoch_end", eta_cb.on_fit_epoch_end)

    results = model.train(**train_params)

    # --- VALIDACIÓN FINAL ---
    print("Entrenamiento finalizado. Evaluando mejor checkpoint...")
    best_weights = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "best.pt")

    # fallback por si no existe best.pt
    if not os.path.exists(best_weights):
        best_weights = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "last.pt")

    model_best = YOLO(best_weights)
    metrics = model_best.val(data=train_params["data"], imgsz=train_params["imgsz"])

    print(f"mAP50     : {metrics.box.map50:.4f}")
    print(f"mAP50-95  : {metrics.box.map:.4f}")
    print(f"Precision : {metrics.box.mp:.4f}")
    print(f"Recall    : {metrics.box.mr:.4f}")
    print(f"Mejor modelo guardado en: {best_weights}")


if __name__ == "__main__":
    main()

### **Retraining with Transfer Learning: VisDrone to FAC Dataset**

In this section, we perform **transfer learning** from the **VisDrone-MOT** model to the new **FAC Dataset**. The goal is to fine-tune the pre-trained **VisDrone** model using a smaller learning rate to preserve the knowledge learned from VisDrone and adapt it to the new dataset.

### 1. **Load Pre-trained Weights from VisDrone-MOT**
The first step is to load the weights from the **VisDrone-MOT**-trained model. This allows us to leverage the previously learned features (such as object detection in aerial images).

# Load pre-trained weights from the VisDrone model

**IMPORTANT:** in `MODEL_WEIGHTS` modify the route of weights (model.pt) for selected `RUN`


In [ ]:

# ---------------- CONFIGURE DIRECTORY OF WEIGHTS ----------------
MODEL_WEIGHTS = 'runs/train_visdrone/....RUN_DIRECTORY..../MODEL.PT'

PROJECT_DIR = 'FAC_finetunning'
RUN_NAME = 'visdrone_to_fac'

train_params = {
    'data': './datasets/dataset_fac/data.yaml',

    'epochs': 70,
    'imgsz': 960,

    'batch': 8,
    'device': 0,

    'close_mosaic': 10,

    # fine tuning
    'lr0': 0.00005,
    'lrf': 0.01,
    'warmup_epochs': 3,

    'freeze': 15,

    # augmentations
    'mosaic': 0.5,
    'mixup': 0.0,
    'hsv_v': 0.2,

    'project': PROJECT_DIR,
    'name': RUN_NAME,

    "save_period": 10,
    'patience': 20,
}

# ---------------- RE-ENTRENAMIENTO ----------------
def main():
    print(f"Iniciando entrenamiento: {RUN_NAME}")
    print(f"Modelo base : {MODEL_WEIGHTS}")
    print(f"Resultados  : {PROJECT_DIR}/{RUN_NAME}\n")

    model = YOLO(MODEL_WEIGHTS)

    # añadir callback
    eta_cb = ETACallback(train_params["epochs"])
    model.add_callback("on_train_start", eta_cb.on_train_start)
    model.add_callback("on_fit_epoch_end", eta_cb.on_fit_epoch_end)

    results = model.train(**train_params)

    # ---------------- VALIDACIÓN FINAL ----------------
    print("\nEntrenamiento finalizado. Evaluando mejor checkpoint...")

    best_weights = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "best.pt")

    if not os.path.exists(best_weights):
        print("⚠️ No se encontró best.pt, usando last.pt")
        best_weights = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "last.pt")

    model_best = YOLO(best_weights)

    metrics = model_best.val(
        data=train_params["data"],
        imgsz=train_params["imgsz"],
        split='val'
    )

    # ---------------- MÉTRICAS ----------------
    print("\n--- MÉTRICAS FINALES ---")
    print(f"mAP50     : {metrics.box.map50:.4f}")
    print(f"mAP50-95  : {metrics.box.map:.4f}")
    print(f"Precision : {metrics.box.mp:.4f}")
    print(f"Recall    : {metrics.box.mr:.4f}")

    print(f"\nMejor modelo guardado en: {best_weights}")

# ---------------- MAIN ----------------
if __name__ == "__main__":
    main()